# Laboratory 4 — Interpretation and design

**Competencies:** C5 (Evaluate), C6 (Create)

You will convert a simulated stress into a device figure of merit, explore the trade-off it 
sits in, and then design a diaphragm to a specification.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))
import numpy as np, matplotlib.pyplot as plt
from memslab.plate import Diaphragm, REFERENCE, REFERENCE_PRESSURE
d, q = REFERENCE, REFERENCE_PRESSURE

print(f'sensitivity = {d.sensitivity(q)*1e6:.3f} uV/V/Pa')
print(f'at 5 V supply, full scale = {d.bridge_output(q)*5*1e3:.0f} mV')


## The trade-off

A thinner diaphragm is more sensitive and closer to failure. That tension is the whole design 
problem. Plot both against thickness and find where they cross your requirements.


In [ ]:
import numpy as np

h_range = np.linspace(10e-6, 40e-6, 60)
S = [Diaphragm(a=d.a, h=h).sensitivity(q)*1e6 for h in h_range]
sigma = [Diaphragm(a=d.a, h=h).edge_stress(q)[0]/1e6 for h in h_range]

fig, ax1 = plt.subplots(figsize=(6.4, 4), dpi=120)
ax1.plot(h_range*1e6, S, color='#1f6f8b'); ax1.set_xlabel('thickness [um]')
ax1.set_ylabel('sensitivity [uV/V/Pa]', color='#1f6f8b')
ax2 = ax1.twinx(); ax2.plot(h_range*1e6, sigma, color='#a33')
ax2.set_ylabel('peak stress [MPa]', color='#a33')
ax1.set_title('Sensitivity and peak stress against thickness'); plt.show()


## Design task

Design a square diaphragm for a 100 kPa full-scale pressure sensor that satisfies **both**:

- sensitivity at least 0.30 uV/V/Pa,
- peak stress at most 120 MPa at full scale, and $w_{max}/h < 0.2$.

Silicon fractures around 7 GPa, but a working design keeps a large margin: 120 MPa is roughly 
a factor of 50, which covers stress concentration at the etched corners, residual stress from 
the process, and fatigue.

Report the design you chose, **which constraint binds**, and what you would change first if 
the specification tightened.


In [ ]:
def evaluate(a_um, h_um, q=REFERENCE_PRESSURE):
    plate = Diaphragm(a=a_um*1e-6, h=h_um*1e-6)
    s = plate.sensitivity(q) * 1e6
    sigma = plate.edge_stress(q)[0] / 1e6
    ratio = plate.w_max(q) / plate.h
    ok = s >= 0.30 and sigma <= 120 and ratio < 0.2
    return {'a_um': a_um, 'h_um': h_um, 'S': round(s, 3),
            'sigma_MPa': round(sigma, 1), 'w/h': round(ratio, 4), 'meets_spec': ok}

# Start here, then search properly.
for h in (20, 25, 30, 35):
    print(evaluate(1000, h))

# TASK: find a design that meets the specification. Do it with a search over a and h,
# not by guessing, and say which constraint stopped you going further.
# TASK: verify your final design with the FE solver, not just the closed form.


## Reflection

1. Which constraint binds your design, and what does that tell you about where the 
   engineering effort should go?
2. Your design used the linear, small-deflection model throughout. Name one effect it omits 
   that would matter in a real device, and say whether it would make your design safer or 
   less safe than predicted.
3. Somebody hands you this design and asks whether they can trust it. What evidence, from 
   all four laboratories, do you give them?
